WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

Swell a bilayer rod until it coils past a full turn and contacts itself.

A uniform bilayer takes uniform curvature, so it closes into a circle after one turn
and its own body would occupy the same space on the next. Self-contact is what stops
that, pushing successive wraps apart into a spiral. The test is run twice, once with
self-contact disabled, so the interpenetration it prevents is visible.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize as spy_opt

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.default_plot_settings)

### Geometry

In [ ]:
rod_length = 1.0
aspect_ratio = 28.0
num_layers = 12
num_elems = 200
num_nodes = num_elems + 1

rod_height = rod_length / aspect_ratio
rod_breadth = 0.7 * rod_height
layer_height = rod_height / num_layers
num_fibers = num_layers

# Layers stacked along d1 only, so every fiber sits at y2 = 0. That leaves kappa_1
# absent from the element energy and the tangent singular out of plane, which the
# planar constraints below remove.
offsets_d1 = jnp.linspace(
  -0.5 * (rod_height - layer_height),
  0.5 * (rod_height - layer_height),
  num_fibers,
)
fiber_offsets = jnp.zeros((num_elems, num_fibers, 2)).at[:, :, 0].set(offsets_d1)
fiber_areas = jnp.full((num_elems, num_fibers), layer_height * rod_breadth)
fiber_heights = jnp.full((num_elems, num_fibers), layer_height)
fiber_thicknesses = jnp.full((num_elems, num_fibers), rod_breadth)

fiber_centroid_moi = jnp.zeros((num_elems, num_fibers, 2))
fiber_centroid_moi = fiber_centroid_moi.at[:, :, 0].set(
  layer_height * rod_breadth**3 / 12.0
)
fiber_centroid_moi = fiber_centroid_moi.at[:, :, 1].set(
  rod_breadth * layer_height**3 / 12.0
)

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda s: jnp.array([s, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, rod_length, num_nodes),
  fiber_offsets=fiber_offsets,
  fiber_areas=fiber_areas,
  fiber_mass_density=jnp.full((num_elems, num_fibers), 1100.0),
  fiber_centroid_moi=fiber_centroid_moi,
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)
contact_radius = float(jnp.max(rod.contact_radii))
element_length = rod_length / num_elems
print(f"contact radius {contact_radius:.5f} m, element length {element_length:.5f} m")

### Material and environment

In [ ]:
shear_modulus_gel, shear_modulus_elastomer = 1.0e6, 1.0e6
chi_gel, chi_elastomer = 0.3, 2.5

solvent = _fe.FlorySolvent(
  bath_temperature=298.0, molar_volume=1.8e-5, ref_chem_potential=0.0
)
num_steps = 100
dry_chem_potential = -1.0e5
wet_chem_potential = -350.0
ramp = jnp.linspace(0.0, 1.0, num_steps + 1)[1:] ** 0.03
chem_potentials = dry_chem_potential + (
  wet_chem_potential - dry_chem_potential
) * ramp

# The bottom half is gel and the top half elastomer, so the section is a bilayer and
# the curvature is uniform along the rod.
pseudodensity = jnp.where(offsets_d1 < 0.0, 1.0, 0.0)
shear_moduli = jnp.broadcast_to(
  shear_modulus_gel * pseudodensity
  + shear_modulus_elastomer * (1.0 - pseudodensity),
  (num_elems, num_fibers),
)
flory_parameters = jnp.broadcast_to(
  chi_gel * pseudodensity + chi_elastomer * (1.0 - pseudodensity),
  (num_elems, num_fibers),
)

In [ ]:
def free_swelling_stretch(flory_param, shear_modulus, chem_potential):
  """Isotropic free-swelling stretch of an unconstrained fiber.

  With no axial load the fiber swells isotropically, so lambda_t = lambda and
  lambda^3 = 1/phi. Substituting into the Flory-Rehner equilibrium leaves a single
  scalar equation in phi.
  """
  gas_const_temp = 8.31446261815324 * solvent.bath_temperature
  coupling = solvent.molar_volume * shear_modulus / gas_const_temp

  def residual(phi):
    return (
      (solvent.ref_chem_potential - chem_potential) / gas_const_temp
      + np.log(1.0 - phi) + phi + flory_param * phi**2
      + coupling * (phi ** (1.0 / 3.0) - phi)
    )

  return spy_opt.brentq(residual, 1e-12, 1.0 - 1e-12) ** (-1.0 / 3.0)


mismatch = free_swelling_stretch(
  chi_gel, shear_modulus_gel, wet_chem_potential
) - free_swelling_stretch(
  chi_elastomer, shear_modulus_elastomer, wet_chem_potential
)
total_turn = 1.5 * mismatch * aspect_ratio
print(f"mismatch          = {mismatch:.4f}")
num_turns = total_turn / (2.0 * np.pi)
print(f"total turn angle  = {total_turn:.3f} rad = {num_turns:.2f} turns")
print(f"coil radius       = {rod_length / total_turn:.4f} m "
      f"({rod_length / total_turn / contact_radius:.1f} rod radii)")

### Self-contact

In [ ]:
# Pairs closer than this many elements apart are excluded. Neighbouring elements are
# always within a contact radius of one another simply by being connected, so too
# small an exclusion fires spurious forces on the undeformed rod. The threshold is
# the number of elements spanning twice the contact radius.
adjacency_exclusion = int(np.ceil(2.0 * contact_radius / element_length)) + 2

# The penalty spring per pair is stiffness * element_length [N/m]. It must exceed the
# structural stiffness it resists, otherwise the rod simply passes through itself. The
# default ratio in `compute_self_contact_stiffness` is below one, which is the wrong
# side of that comparison for a penalty formulation.
self_contact_stiffness = _contact.compute_self_contact_stiffness(
  rod=rod,
  youngs_modulus=3.0 * shear_modulus_gel,
  stiffness_ratio=0.50,
)
self_contact = _contact.SelfContact.from_rod(
  rod=rod,
  stiffness=self_contact_stiffness,
  adjacency_exclusion=adjacency_exclusion,
  smoothing_fraction=0.05,
)
print(f"adjacency exclusion = {adjacency_exclusion} elements")
print(f"contact pairs       = {self_contact.num_contact_pairs}")
print(f"penalty stiffness   = {self_contact_stiffness:.4e} Pa")

### Boundary conditions and solve

In [ ]:
clamped_dofs = (
  _utils.Translation.X, _utils.Translation.Y, _utils.Translation.Z,
  _utils.Rotation.THETA_X, _utils.Rotation.THETA_Y, _utils.Rotation.THETA_Z,
)
bc = _bc.process_boundary_conditions(
  rod,
  [
    _bc.DirichletBC(
      values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
      node_targets=[np.array([0]) for _ in range(num_steps)],
      name=f"clamp_{dof}",
    )
    for dof in clamped_dofs
  ]
  + [
    _bc.DirichletBC(
      values=(dof, [np.zeros(num_nodes) for _ in range(num_steps)]),
      node_targets=[np.arange(num_nodes) for _ in range(num_steps)],
      name=f"planar_{dof}",
    )
    for dof in (_utils.Translation.Z, _utils.Rotation.THETA_Y)
  ],
)
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 200, "threshold": 1e-6, "abs_threshold": 1e-6, "lam_min": 0.01,
    "line_search_max_iter": 15, "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6, "line_search_armijo_c": 1e-4,
  },
  linear_settings={"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-10},
)


def solve_coil(with_self_contact: bool):
  """Swell the bilayer with self-contact either active or disabled."""
  problem = _fe.CosseratProblem(
    rod=rod,
    bc=bc,
    nr_solver=nr_solver,
    solvent=solvent,
    self_contact=self_contact if with_self_contact else None,
    bisection_settings={
      "lower": 1e-6, "upper": 1.0 - 1e-6, "tol": 1e-6, "max_iter": 200
    },
  )
  u_star, _ = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=shear_moduli,
    flory_parameters=flory_parameters,
    chem_potentials=chem_potentials,
  )
  return u_star

In [ ]:
u_free = solve_coil(with_self_contact=False)
u_contact = solve_coil(with_self_contact=True)

### Verification

In [ ]:
def min_pair_separation(u: jnp.ndarray) -> np.ndarray:
  """Smallest distance between any two non-adjacent element segments.

  Self-contact must keep every excluded pair at least the sum of their contact radii
  apart. Any value below that is penetration, and a value near zero means the rod has
  passed through itself.
  """
  coords, _, _ = _mesher.extract_coords_and_director_from_deformation(rod, u)
  coords = np.asarray(coords)
  pairs = self_contact.pairs
  distances = np.array([
    float(
      _contact._segment_segment_distance(
        p0=jnp.asarray(coords[a]), p1=jnp.asarray(coords[a + 1]),
        q0=jnp.asarray(coords[b]), q1=jnp.asarray(coords[b + 1]),
      )
    )
    for a, b in pairs
  ])
  return distances


threshold = 2.0 * contact_radius
for label, u in [("no self-contact", u_free), ("self-contact", u_contact)]:
  distances = min_pair_separation(u)
  print(f"{label:16s} min pair separation = {distances.min():.5f} m   "
        f"threshold {threshold:.5f}   "
        f"{'PENETRATES' if distances.min() < 0.5 * threshold else 'ok'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
cases = [("no self-contact", u_free), ("self-contact", u_contact)]
for ax, (label, u) in zip(axes, cases):
  coords, _, _ = _mesher.extract_coords_and_director_from_deformation(rod, u)
  coords = np.asarray(coords)
  ax.plot(coords[:, 0], coords[:, 1], lw=1.0, color="0.3")
  # Drawn at the contact radius, so the bands touch exactly when contact activates.
  ax.plot(coords[:, 0], coords[:, 1], lw=0.0)
  circle_theta = np.linspace(0.0, 2.0 * np.pi, 40)
  for node in range(0, num_nodes, 8):
    ax.plot(
      coords[node, 0] + contact_radius * np.cos(circle_theta),
      coords[node, 1] + contact_radius * np.sin(circle_theta),
      lw=0.4, color="tab:blue", alpha=0.5,
    )
  ax.set_aspect("equal")
  ax.set_title(label, fontsize=10)
  ax.set_xlabel("$x$ [m]")
  ax.set_ylabel("$y$ [m]")
fig.tight_layout()
plt.show()

In [ ]:
# Spacing between successive wraps. With self-contact fully seated it settles at twice
# the contact radius, since that is the closest the centrelines can approach.
coords_contact = np.asarray(
  _mesher.extract_coords_and_director_from_deformation(rod, u_contact)[0]
)
radius_from_centre = np.linalg.norm(
  coords_contact[:, :2] - coords_contact[:, :2].mean(axis=0), axis=1
)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(np.asarray(rod.arc_coords), radius_from_centre, lw=1.5)
ax.set_xlabel(r"reference arc coordinate $\hat{s}$ [m]")
ax.set_ylabel("radius from coil centre [m]")
plt.show()

print(f"wrap spacing (max - min radius) = "
      f"{radius_from_centre.max() - radius_from_centre.min():.5f} m")
print(f"twice the contact radius        = {threshold:.5f} m")